# Phase 3 (Rhetorical Roles: CL + IT) — Run on Phase-1 **Encoder-Decoder model 01** (bidirectional encoder), with **IA³**

Config-driven, like the Phase-2 notebooks: edit **Cell 2** only. This is the third Phase-3 run of your plan: the **encoder** of the Phase-1 encoder-decoder, adapted with **IA³**, the lightest technique in the list (about 15 thousand trainable parameters in the encoder). The data handling, windows, metrics, bias tuning and submission writer are the same as in the two decoder notebooks, so the three runs are directly comparable.

### The task in one paragraph
Every document is an ordered list of sentences; predict **one of 13 rhetorical-role labels per sentence** (order read from `label_vocab.json`). Metric: **macro-F1**, so rare roles weigh as much as `Fact`. Model = Phase-1 **encoder** + **mean-pool over the tokens of each sentence** + `Linear(512 → 13)`: exactly the Module 2.4 recipe (a BERT-style encoder with one extra output layer), applied per sentence instead of per token.

### Why this model is the natural fit for sentence classification
| | Decoders 01 / 02 / 03 | **Encoder (this notebook)** |
|---|---|---|
| context for a sentence | left only (echo trick or unmasking needed) | **both sides, natively**: full self-attention (`encoder_causal=False`) |
| Phase-1 training signal | next-token prediction | **masked-LM** (15 %, 80/10/10) on the encoder, plus the seq2seq loss through cross-attention |
| used at fine-tuning time | the whole model | **the encoder only**; decoder, cross-attention and LM head are dropped |

Size: embeddings 16.38M + 6 encoder blocks 19.27M = **35.66M** (the full Phase-1 model is 61.49M). **KD warning:** the capstone requires the teacher to be *larger* than the student, and 35.66M is below the default 36.58M Dec-03 student, so this run **cannot** be that student's teacher (use a ≈55M decoder run).

### IA³ — *Infused Adapter by Inhibiting and Amplifying Inner Activations* (Liu et al. 2022)
IA³ freezes the whole backbone and learns **one scaling vector per site**, initialised to 1 (so the model starts exactly at the Phase-1 function; asserted at build time):

* `k ← l_k ⊙ (W_k x)` and `v ← l_v ⊙ (W_v x)` in every attention layer (`l_k, l_v ∈ R^512`),
* `h_ff ← l_ff ⊙ (silu(W_gate x) ⊙ W_up x)` before `down_proj` in every SwiGLU block (`l_ff ∈ R^1408`).

That is `512 + 512 + 1408 = 2 432` numbers per block, **14 592** for the six encoder blocks, plus the 13-way head (6 669). Compare with LoRA at r = 16 on the 7 linear types of the same six blocks (about 0.95M parameters): IA³ is *diagonal* (it can only amplify or inhibit existing feature channels) while LoRA adds a dense low-rank update. Because of that, expect IA³ to behave like "a linear probe on the pretrained features plus a per-channel re-weighting"; whether the MLM-pretrained encoder features are already good enough for rhetorical roles is exactly the question *"are the Phase-1 representations useful?"* from the capstone's analysis list. Its larger learning rate (3e-3, as in the IA³ paper's setting) is deliberate: the vectors start at 1 and only need small moves.

### `METHOD`
| `METHOD` | What is trained |
|---|---|
| `"ia3"` | the IA³ vectors at the sites in `IA3_SITES` + head (**default**) |
| `"full"` | encoder + embeddings + head (full-fine-tuning reference) |
| `"lora"` / `"lorma"` | adapters on the 7 Linear types of every encoder block + head (same code as the decoder notebooks) |
| `"norm_only"` | RMSNorm gains + head |

### Windows
The encoder was trained on 768-token inputs (`encoder_len`) and its RoPE table covers 1024 positions, so the default `WINDOW_TOKENS=700` stays inside the trained length (the notebook warns above 768). `CONTEXT_MODE="sentence"` (one sentence per sequence) is the no-context control; `WINDOW_TOKENS` is the context-size knob. The encoder is bidirectional, so no echo trick and no `ATTN_MODE` choice exists here.

### Class imbalance, domains, outputs
Same as the decoder notebooks: class-weighted cross-entropy (`sqrt_inv`); a per-class logit bias tuned on dev **and kept only if a 2-fold document-level check shows a held-out gain**; CL and IT trained jointly with a text tag; selection score = mean of the per-domain macro-F1; one `prediction.json` with all CL and IT test documents. Outputs go to `WORK_DIR/runs/<RUN_NAME>/`: `run_log.json`, `rr_*.pt`, `submission/submission.zip`, optionally `teacher_softlabels_*.pt`.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field, fields
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p3_encdec_ia3"
METHOD     = "ia3"            # "ia3" | "full" | "lora" | "lorma" | "norm_only"
SMOKE_TEST = False            # True: ~250 steps + a few dev docs, dev check only, NO submission written
ATTN_MODE  = "bidir"          # FIXED: the encoder is bidirectional by construction (kept as a constant for the shared cells)
IA3_SITES  = ("k", "v", "ff") # METHOD="ia3": which activations get a learned scaling vector (keys, values, FFN hidden)

# ---- Phase-1 checkpoint (written by caps_enc_dec_architecture_01.ipynb; must have encoder_causal=False) ----------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # files are Phase_3_RR/{Train,Test}/...
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_3_RR/ (checked first)
DOMAINS    = ["CL", "IT"]                         # both domains are trained jointly; use ["CL"] or ["IT"] for a single-domain run
DOMAIN_TAG = True                                 # prepend a short text tag naming the domain
FINAL_FIT  = False                                # True: train on train+dev, NO validation (use after the dev run, EPOCHS = best epoch)

# ---- Context (the sentence-level vs document-level context ablation) ----------------------------
CONTEXT_MODE   = "window"                         # "window" (many sentences per sequence) | "sentence" (one sentence per sequence, no context)
WINDOW_TOKENS  = 700                              # max tokens per sequence; the encoder was trained on 768, its RoPE table covers 1024
WINDOW_OVERLAP = 0.5                              # fraction of a window that the next window repeats
MAX_SENT_TOK   = 160                              # each sentence is truncated to this many tokens
EDGE_WEIGHT    = True                             # at inference weigh a sentence's prediction by its distance from the window edge

# ---- Training -----------------------------------------------------------------------------
EPOCHS       = 8                                  # IA3 trains very few parameters and converges slowly; use 6 for the other methods
BATCH_WINDOW = 8                                  # windows per micro-batch (CONTEXT_MODE="window")
BATCH_SENT   = 64                                 # sentences per micro-batch (CONTEXT_MODE="sentence")
ACCUM        = 2                                  # gradient accumulation
WARMUP_FRAC  = 0.06
GRAD_CLIP    = 1.0
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "ia3":       dict(lr=3e-3, head_lr=1e-3, wd=0.0),
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),
    "lorma":     dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, variant="plus"),   # "plus" (LoRMA+) | "pi" (LoRMA_pi)
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")   # LoRA and LoRMA (encoder blocks)
HEAD_DROPOUT    = 0.1
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
LABEL_SMOOTHING = 0.0

# ---- Per-class logit bias tuned on dev (macro-F1 post-processing) -------------------------------
BIAS_TUNE     = True
BIAS_MIN_GAIN = 0.005                             # keep the tuned bias only if a 2-fold document-level check on dev gains this much macro-F1
BIAS_FINAL    = None                              # used when FINAL_FIT=True: paste the bias list printed by the dev run (or None)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_VAL_DOCS = 250, 10

# ---- Knowledge distillation: this encoder (35.66M) is SMALLER than the default 36.58M student, so it cannot be its teacher ----------
EXPORT_SOFT_LABELS = False        # only useful for a smaller student (STUDENT_KEEP_LAYERS <= 6 blocks in the KD notebook)

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE == "bidir" and CONTEXT_MODE in ("window", "sentence")
assert set(IA3_SITES) <= {"k", "v", "ff"} and len(IA3_SITES) > 0
assert 0.0 <= WINDOW_OVERLAP < 1.0 and WINDOW_TOKENS <= 1024, "sequences longer than 1024 exceed the RoPE table"
assert WINDOW_TOKENS >= MAX_SENT_TOK + 16, "a window must at least hold one (truncated) sentence plus the tags"
SEQ_LEN = WINDOW_TOKENS                           # (used by the checkpoint-loading cell)
BATCH = BATCH_WINDOW if CONTEXT_MODE == "window" else BATCH_SENT
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | encoder-only RR | CONTEXT_MODE={CONTEXT_MODE} (window={WINDOW_TOKENS}, overlap={WINDOW_OVERLAP}) | "
      f"domains={DOMAINS} | final_fit={FINAL_FIT} | smoke={SMOKE_TEST}")

RUN p3_encdec_ia3 | METHOD=ia3 | encoder-only RR | CONTEXT_MODE=window (window=700, overlap=0.5) | domains=['CL', 'IT'] | final_fit=False | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# NER text must reach the tokenizer UNCHANGED (no clean_text / normalisation), otherwise character offsets drift.
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_enc_dec_architecture_01.ipynb` (its Cell 3: model, checkpoint helpers)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

Default config params: total=61.49M (embedding 16.38M)


## Load the Phase-1 checkpoint

In [6]:
# Cell 12 — Load the Phase-1 encoder-decoder checkpoint and sanity-check its ENCODER
lm, lm_cfg, _ck = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)       # refuses a causal-encoder checkpoint
del _ck
pc = count_params(lm)
n_enc = sum(p.numel() for p in lm.encoder.parameters())                              # embeddings (shared) + encoder blocks + final norm
print(f"Phase-1 model | arch={lm_cfg.arch_name} | enc layers={lm_cfg.n_encoder_layers} dec layers={lm_cfg.n_decoder_layers} "
      f"d_model={lm_cfg.d_model} pos={lm_cfg.pos_type} | total {pc['total']/1e6:.2f}M | ENCODER-ONLY (used for NER) {n_enc/1e6:.2f}M")
assert SEQ_LEN <= lm_cfg.max_seq_len, "window longer than the RoPE table of this checkpoint"
if SEQ_LEN > lm_cfg.encoder_len:
    print(f"WARNING: sequence length {SEQ_LEN} > the encoder's trained length {lm_cfg.encoder_len} (RoPE extrapolation, expect some degradation)")

# Sanity: if the weights did not load properly, masked-token prediction would sit near ln(V) ~ 10.4.
MASK_ID = getattr(tokenizer, "special_to_id", {}).get("<mask>", 6)
if lm_cfg.mlm_head:
    _sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
    _x = torch.tensor([[BOS_ID] + tokenizer.encode(_sent)], device=device)
    _pos = [p for p in (4, 9, 13) if p < _x.size(1) - 1]
    _xm = _x.clone(); _xm[0, _pos] = MASK_ID
    _sel = torch.zeros_like(_xm, dtype=torch.bool); _sel[0, _pos] = True
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        _h = lm.encode(_xm, torch.ones_like(_xm))
        _lg = lm.mlm_logits_at(_h, _sel)
    _l = F.cross_entropy(_lg.float(), _x[0, _pos]).item()
    print(f"masked-LM loss on a legal sentence (3 masked tokens): {_l:.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
    assert _l < 9.0, "checkpoint looks untrained / mis-loaded"
else:
    print("this checkpoint has no MLM head; skipping the masked-LM sanity check")

Loaded /kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
Phase-1 model | arch=encdec_bidir_mlm_v2 | enc layers=6 dec layers=6 d_model=512 pos=rope | total 61.49M | ENCODER-ONLY (used for NER) 35.66M
masked-LM loss on a legal sentence (3 masked tokens): 0.117  (ln V = 10.37)


## Phase-3 data: labels, documents, tokenisation
Documents are `{"id", "text": [sentences], "labels": [labels]}`. The loader also accepts a few equivalent layouts (dict keyed by id, `sentences` instead of `text`) and **fails loudly** with the offending example if a label is not in `label_vocab.json` or if the numbers of sentences and labels differ. If the schema printed below is not what you expect, paste it to Claude.

In [7]:
# Cell 13 — Phase-3 data: label vocabulary, documents (CL / IT, train / dev / test)
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

P3 = "Phase_3_RR"
SPEC_LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
               "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"]

def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_3_RR folder and set LOCAL_DATA_DIR.") from ex

def load_json(rel):
    return json.loads(fetch(rel).read_text(encoding="utf-8"))

def parse_label_vocab(raw):
    """label_vocab.json -> ordered list of label names (accepts a list, {label: id}, {id: label} or {"labels": [...]})."""
    if isinstance(raw, dict) and "labels" in raw and isinstance(raw["labels"], (list, dict)):
        raw = raw["labels"]
    if isinstance(raw, list):
        if all(isinstance(x, str) for x in raw): return list(raw)
        if all(isinstance(x, dict) for x in raw):
            out = []
            for x in raw:
                k = next((x[k] for k in ("label", "name", "role") if isinstance(x.get(k), str)), None)
                if k is None: raise ValueError(f"cannot parse label_vocab entry {x!r}")
                out.append(k)
            return out
    if isinstance(raw, dict):
        if all(isinstance(v, int) and not isinstance(v, bool) for v in raw.values()):
            return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
        if all(isinstance(k, str) and k.isdigit() for k in raw) and all(isinstance(v, str) for v in raw.values()):
            return [raw[k] for k in sorted(raw, key=int)]
        return list(raw)
    raise ValueError(f"cannot parse label_vocab.json: {type(raw).__name__}")

LABELS = parse_label_vocab(load_json(f"{P3}/Train/label_vocab.json"))
if set(LABELS) != set(SPEC_LABELS):
    print("WARNING: label_vocab.json differs from the 13 labels in the capstone document:", LABELS)
NL = len(LABELS); L2I = {l: i for i, l in enumerate(LABELS)}
print(NL, "labels:", LABELS)

def parse_docs(raw, domain, has_labels):
    """-> list of {uid, id, domain, sents, y|None}. uid = 'DOMAIN::id' (unique across domains, used internally only)."""
    if isinstance(raw, dict):
        items = []
        for k, v in raw.items():
            if isinstance(v, list): v = {"text": v}
            if not isinstance(v, dict): raise ValueError(f"unexpected document layout for key {k!r}: {type(v).__name__}")
            v = dict(v); v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        did = str(next((e[k] for k in ("id", "doc_id", "document_id") if k in e), i))
        sents = e.get("text", e.get("sentences"))
        if not isinstance(sents, list):
            raise ValueError(f"{domain} doc {did}: expected 'text' to be a list of sentences, got {type(sents).__name__}. Keys: {list(e)}")
        sents = [s if isinstance(s, str) else str(s.get("text", "")) for s in sents]
        y = None
        if has_labels:
            labs = e.get("labels", e.get("label", e.get("roles")))
            if not isinstance(labs, list) or len(labs) != len(sents):
                raise ValueError(f"{domain} doc {did}: {len(sents)} sentences but labels = "
                                 f"{None if not isinstance(labs, list) else len(labs)}. Keys: {list(e)}")
            bad = [l for l in labs if l not in L2I]
            if bad: raise ValueError(f"{domain} doc {did}: label(s) {sorted(set(bad))[:5]} not in label_vocab.json {LABELS}")
            y = np.asarray([L2I[l] for l in labs], dtype=np.int64)
        docs.append(dict(uid=f"{domain}::{did}", id=did, domain=domain, sents=sents, y=y))
    return docs

def load_split(domain, split):
    rel = f"{P3}/{'Test' if split == 'test' else 'Train'}/{domain}_{split}.json"
    raw = load_json(rel)
    first = raw[0] if isinstance(raw, list) and raw else (next(iter(raw.values())) if isinstance(raw, dict) and raw else None)
    print(f"  {rel}: {type(raw).__name__} of {len(raw)} | first item keys: {list(first) if isinstance(first, dict) else type(first).__name__}")
    return parse_docs(raw, domain, has_labels=(split != "test"))

TRAIN_ALL = {dom: load_split(dom, "train") for dom in DOMAINS}
DEV_ALL   = {dom: load_split(dom, "dev")   for dom in DOMAINS}
TEST_ALL  = {dom: load_split(dom, "test")  for dom in DOMAINS}

for dom in DOMAINS:
    for name, split in (("train", TRAIN_ALL), ("dev", DEV_ALL), ("test", TEST_ALL)):
        ds = split[dom]; print(f"{dom} {name:5s}: {len(ds):4d} docs | {sum(len(d['sents']) for d in ds):6d} sentences")
    cnt = Counter(LABELS[i] for d in TRAIN_ALL[dom] for i in d["y"])
    print(f"  {dom} train label counts:", dict(cnt.most_common()))
_test_ids = [d["id"] for dom in DOMAINS for d in TEST_ALL[dom]]
if len(set(_test_ids)) != len(_test_ids):
    print("WARNING: test document ids repeat across the domains; the single prediction.json cannot hold both -> paste this to Claude.")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

13 labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  Phase_3_RR/Train/CL_train.json: list of 40 | first item keys: ['id', 'text', 'labels']


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

  Phase_3_RR/Train/IT_train.json: list of 40 | first item keys: ['id', 'text', 'labels']


... (4 outputs left out)

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

  Phase_3_RR/Test/CL_test.json: dict of 5 | first item keys: list


IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

  Phase_3_RR/Test/IT_test.json: dict of 5 | first item keys: list
CL train:   40 docs |  10593 sentences
CL dev  :    5 docs |   1396 sentences
CL test :    5 docs |   1339 sentences
  CL train label counts: {'Fact': 2979, 'RatioOfTheDecision': 2856, 'PrecedentReliedUpon': 1305, 'ArgumentRespondent': 996, 'ArgumentPetitioner': 976, 'RulingByLowerCourt': 438, 'Statute': 367, 'Dissent': 284, 'PrecedentNotReliedUpon': 209, 'RulingByPresentCourt': 113, 'None': 53, 'Issue': 17}
IT train:   40 docs |   6252 sentences
IT dev  :    5 docs |    746 sentences
IT test :    5 docs |    858 sentences
  IT train label counts: {'RatioOfTheDecision': 1445, 'Fact': 1119, 'Statute': 732, 'PrecedentReliedUpon': 640, 'ArgumentPetitioner': 637, 'RulingByLowerCourt': 436, 'PrecedentNotReliedUpon': 360, 'ArgumentRespondent': 352, 'None': 244, 'RulingByPresentCourt': 194, 'Issue': 90, 'PrecedentOverruled': 3}


In [8]:
# Cell 14 — Tokenise the sentences (shared byte-level BPE), domain tags, delimiter, cache on disk
DOMAIN_TEXT = {"CL": "Corporate Law\n", "IT": "Information Technology\n"}
NL_IDS  = list(tokenizer.encode("\n"))                       # sentence delimiter (the Phase-1 corpus kept newlines)
DOM_IDS = {dom: (list(tokenizer.encode(DOMAIN_TEXT.get(dom, dom + "\n"))) if DOMAIN_TAG else []) for dom in DOMAINS}
NLN = len(NL_IDS)
BASE_LEN = {dom: 1 + len(DOM_IDS[dom]) for dom in DOMAINS}   # [BOS] + domain tag
assert NLN >= 1
print("delimiter ids:", NL_IDS, "| domain tag ids:", {k: v for k, v in DOM_IDS.items()})

def tokenize_docs(docs, cache_name):
    """Adds d['tok'] (list of int32 arrays, each truncated to MAX_SENT_TOK, never empty) and d['full_len']. Cached on disk."""
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"{cache_name}_v{VOCAB_SIZE}_m{MAX_SENT_TOK}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    changed = False
    for d in tqdm(docs, desc=f"tokenise {cache_name}", leave=False):
        hit = cache.get(d["uid"])
        if hit is None or len(hit[0]) != len(d["sents"]):
            toks, full = [], []
            for s in d["sents"]:
                ids = list(tokenizer.encode(s)); full.append(len(ids))
                ids = ids[:MAX_SENT_TOK] if ids else NL_IDS[:1]          # an empty sentence still needs one token to pool over
                toks.append(np.asarray(ids, dtype=np.int32))
            hit = (toks, full); cache[d["uid"]] = hit; changed = True
        d["tok"], d["full_len"] = hit
    if changed:
        pickle.dump(cache, open(cp, "wb"))
    return docs

def token_stats(docs, title):
    lens = np.concatenate([np.asarray(d["full_len"]) for d in docs]) if docs else np.zeros(1)
    print(f"{title}: {len(lens)} sentences | tokens/sentence mean {lens.mean():.1f}, median {np.median(lens):.0f}, p95 {np.percentile(lens, 95):.0f}, "
          f"max {lens.max()} | truncated at {MAX_SENT_TOK}: {100 * (lens > MAX_SENT_TOK).mean():.2f}%")

delimiter ids: [133] | domain tag ids: {'CL': [5390, 5865, 3111, 133], 'IT': [1107, 4156, 3691, 133]}


## Windows, batches, metrics, bias tuning (pure python / numpy)
`make_windows` packs consecutive sentences into sequences of at most `WINDOW_TOKENS` tokens; the next window starts after the first `1 − WINDOW_OVERLAP` of the previous one, so every sentence is covered and most sentences appear in two windows. During training the first window of each document gets a random shorter budget every epoch, which shifts all window boundaries.

In [9]:
# Cell 15 — Windows and batch layout
def make_windows(d, budget, overlap, first_budget=None):
    """Sentence index ranges (a, b) so that  BASE + sum(len) + (b-a-1)*NLN <= budget  (at least one sentence per window).
    The next window starts after the first (1-overlap) of this window's tokens; the last window always ends at the last sentence."""
    lens = [len(t) for t in d["tok"]]; base = BASE_LEN[d["domain"]]; n = len(lens)
    out, a, first = [], 0, True
    while a < n:
        bud = first_budget if (first and first_budget is not None) else budget
        tot, b = base, a
        while b < n:
            add = lens[b] + (NLN if b > a else 0)
            if b > a and tot + add > bud: break
            tot += add; b += 1
        out.append((a, b))
        if b >= n: break
        target, adv, a2 = (1.0 - overlap) * (tot - base), 0, a
        while a2 < b - 1 and adv < target:
            adv += lens[a2] + NLN; a2 += 1
        a = max(a2, a + 1); first = False
    return out

def doc_windows(d, rng=None):
    """Training (rng given): randomised first window. Inference (rng None): deterministic."""
    n = len(d["tok"])
    if CONTEXT_MODE == "sentence":
        return [(i, i + 1) for i in range(n)]
    fb = int(WINDOW_TOKENS * rng.uniform(0.4, 1.0)) if rng is not None else None
    return make_windows(d, WINDOW_TOKENS, WINDOW_OVERLAP, fb)

def build_batch(items, docs, with_labels=True):
    """items: [(doc_index, a, b)].  Returns ids, mask, seg, y, owners, n_sent.
       seg[b, t] = global sentence number (0..n_sent-1) of the token, or -1 for BOS / domain tag / delimiters / padding.
       owners[g] = (doc_index, sentence_index, a, b) for global sentence g."""
    seqs, segs, ys, owners, g = [], [], [], [], 0
    for di, a, b in items:
        d = docs[di]; dom = DOM_IDS[d["domain"]]
        seq = [BOS_ID] + dom; seg = [-1] * len(seq)
        for j in range(a, b):
            if j > a:
                seq += NL_IDS; seg += [-1] * NLN
            t = d["tok"][j].tolist()
            seq += t; seg += [g] * len(t)
            owners.append((di, j, a, b))
            if with_labels: ys.append(int(d["y"][j]))
            g += 1
        assert len(seq) <= 1024, f"sequence of {len(seq)} tokens exceeds the Phase-1 context"
        seqs.append(seq); segs.append(seg)
    L = max(len(s) for s in seqs)
    ids  = np.full((len(seqs), L), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(seqs), L), dtype=np.int64)
    sg   = np.full((len(seqs), L), -1, dtype=np.int64)
    for i, (s, q) in enumerate(zip(seqs, segs)):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1; sg[i, :len(q)] = q
    y = np.asarray(ys, dtype=np.int64) if with_labels else None
    return ids, mask, sg, y, owners, g

In [10]:
# Cell 16 — Metrics (macro-F1, per domain), confusion, per-class bias tuning
def confusion(y, p):
    cm = np.zeros((NL, NL), dtype=np.int64); np.add.at(cm, (y, p), 1); return cm

def scores_from_cm(cm):
    """cm[gold, pred].  macro_f1 averages over labels that occur in gold OR prediction (sklearn's default);
    macro_f1_gold averages over labels present in the gold only."""
    tp = np.diag(cm).astype(np.float64); sup = cm.sum(1).astype(np.float64); npred = cm.sum(0).astype(np.float64)
    P = np.divide(tp, npred, out=np.zeros(NL), where=npred > 0)
    R = np.divide(tp, sup,   out=np.zeros(NL), where=sup > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(NL), where=(P + R) > 0)
    union, gold = (sup + npred) > 0, sup > 0
    return dict(P=P, R=R, F=F, support=sup, cm=cm, accuracy=float(tp.sum() / max(cm.sum(), 1)),
                macro_f1=float(F[union].mean()), macro_p=float(P[union].mean()), macro_r=float(R[union].mean()),
                macro_f1_gold=float(F[gold].mean()))

def evaluate(docs, probs, bias=None):
    """docs with gold labels + per-doc [n_sent, NL] probabilities -> {domain: scores, 'ALL': scores, 'select': mean of per-domain macro-F1}."""
    ys, ps = defaultdict(list), defaultdict(list)
    for d, p in zip(docs, probs):
        lp = np.log(np.maximum(p, 1e-8)) + (0.0 if bias is None else np.asarray(bias))
        ys[d["domain"]].append(d["y"]); ps[d["domain"]].append(lp.argmax(1))
    out = {dom: scores_from_cm(confusion(np.concatenate(ys[dom]), np.concatenate(ps[dom]))) for dom in ys}
    out["ALL"] = scores_from_cm(confusion(np.concatenate([np.concatenate(v) for v in ys.values()]),
                                          np.concatenate([np.concatenate(v) for v in ps.values()])))
    out["select"] = float(np.mean([out[dom]["macro_f1"] for dom in ys]))
    return out

def tune_bias(docs, probs, grid=np.arange(-1.5, 1.51, 0.25), passes=2):
    """Coordinate ascent on a per-class additive log-probability bias, objective = mean per-domain macro-F1 on these docs."""
    LP = np.concatenate([np.log(np.maximum(p, 1e-8)) for p in probs])
    Y  = np.concatenate([d["y"] for d in docs])
    DM = np.concatenate([[d["domain"]] * len(d["y"]) for d in docs])
    doms = sorted(set(DM.tolist()))
    def obj(b):
        pred = (LP + b).argmax(1)
        return float(np.mean([scores_from_cm(confusion(Y[DM == dm], pred[DM == dm]))["macro_f1"] for dm in doms]))
    b = np.zeros(NL); base = best = obj(b)
    for _ in range(passes):
        for c in range(NL):
            keep = b[c]
            for v in grid:
                b2 = b.copy(); b2[c] = v; s = obj(b2)
                if s > best + 1e-9: best, keep = s, v
            b[c] = keep
    return b, base, best

def cv_bias_gain(docs, probs, folds=2):
    """Honest check of the bias tuning: tune on half of the dev DOCUMENTS, score the other half (and vice versa); mean held-out gain."""
    gains = []
    for f in range(folds):
        tr = [i for i in range(len(docs)) if i % folds != f]; te = [i for i in range(len(docs)) if i % folds == f]
        b, _, _ = tune_bias([docs[i] for i in tr], [probs[i] for i in tr])
        base  = evaluate([docs[i] for i in te], [probs[i] for i in te])["select"]
        tuned = evaluate([docs[i] for i in te], [probs[i] for i in te], b)["select"]
        gains.append(tuned - base)
    return float(np.mean(gains))

def top_confusions(cm, k=10):
    pairs = [(int(cm[i, j]), LABELS[i], LABELS[j]) for i in range(NL) for j in range(NL) if i != j and cm[i, j] > 0]
    return sorted(pairs, reverse=True)[:k]

## Encoder model, adaptation methods (IA³ / full / LoRA / LoRMA / norm-only), training
LoRA and LoRMA are the Phase-2 layers, unchanged. **IA³ is new** (two small wrapper modules). The training loop and the inference code are shared with the decoder notebooks.

In [11]:
# Cell 17 — Encoder RR model and the adaptation methods
import copy

class EncRRModel(nn.Module):
    """Phase-1 ENCODER (embeddings -> 6 bidirectional blocks -> final RMSNorm), mean over the tokens of each sentence, dropout,
    Linear(d_model -> n_labels). The Module 2.4 recipe at sentence level. The decoder is not used."""
    def __init__(self, encoder, d_model, n_labels, dropout):
        super().__init__()
        self.encoder = encoder
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask, seg, n_sent):
        h = self.encoder(input_ids, attention_mask=attention_mask).float()      # [B, T, d]; attention_mask = key padding mask
        valid = seg >= 0
        idx = seg[valid]                                                         # global sentence number of every sentence token
        pooled = h.new_zeros(n_sent, h.size(-1)).index_add_(0, idx, h[valid])   # sum of the token vectors of each sentence
        cnt = torch.bincount(idx, minlength=n_sent).clamp_min(1).unsqueeze(1).to(pooled.dtype)
        return self.head(self.drop(pooled / cnt))                                # [n_sent, n_labels]


# ---- (c) LoRA (additive) --------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))).  W frozen; B = 0 at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


# ---- (d) LoRMA (multiplicative; Bihany, Patel & Modi, ACL Findings 2025) -----------------------------------------------------
_PI_IDX = {}
def _pi_index(d, dev):
    """idx[i, j] = (j - i) mod d.  gather(M, 1, idx)[i, j] = M[i, (j - i) mod d]  ==  row i of M rotated right by i."""
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    """h = I(B A) (W0 x).  W0 frozen.  A in R^{r x d_out}, B in R^{d_out x r}, d_out = base.out_features.
       'plus' (LoRMA+):  I(BA) = I_d + (alpha/r) BA, computed as y + (alpha/r) B(A y); B = 0 at start.
       'pi'   (LoRMA_pi): row i of BA rotated cyclically by i; B[:,0]=1, A[0,0]=1, rest of A = 0 => I(BA) = I_d at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)                                              # W0 x (frozen); the multiplier acts on the OUTPUT space
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))
        return y @ Tm.t()


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n



# ---- (e) IA3 (Liu et al. 2022): learned per-channel scaling of keys, values and the FFN hidden activation -----------------------
class IA3Out(nn.Module):
    """y = l * (W x)   with l in R^{out_features}, initialised to 1.  Rescales the OUTPUT channels (keys / values).
    (Keys are scaled before RoPE rotates them; the relative-position property of RoPE only needs q and k to be rotated, so it survives.)"""
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        y = self.base(x)
        return y * self.l.to(y.dtype)


class IA3In(nn.Module):
    """y = W (l * x)   with l in R^{in_features}, initialised to 1.  Used on down_proj: scaling its INPUT is scaling the FFN hidden
    activation silu(gate(x)) * up(x), i.e. the l_ff of the IA3 paper."""
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))


def build_rr_model():
    """Fresh copy of the Phase-1 encoder every call (several runs in one session do not contaminate each other)."""
    enc = copy.deepcopy(lm.encoder).cpu()
    ref = copy.deepcopy(enc) if METHOD in ("lora", "lorma", "ia3") else None   # untouched copy for the identity check
    model = EncRRModel(enc, lm_cfg.d_model, NL, HEAD_DROPOUT)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # encoder + embeddings + head all trainable
    elif METHOD in ("lora", "lorma", "ia3"):
        for p in enc.parameters(): p.requires_grad = False
        if METHOD == "lora":
            n = apply_adapter(enc, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"]))
            print(f"LoRA applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        elif METHOD == "lorma":
            n = apply_adapter(enc, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
            print(f"LoRMA[{hp['variant']}] applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        else:
            n_out = apply_adapter(enc, tuple(t for s, t in (("k", "k_proj"), ("v", "v_proj")) if s in IA3_SITES), IA3Out)
            n_in = apply_adapter(enc, ("down_proj",), IA3In) if "ff" in IA3_SITES else 0
            print(f"IA3 applied: {n_out} output-scaling vectors (keys/values) + {n_in} FFN-hidden vectors | sites={IA3_SITES}")
        # The adapted encoder must equal the Phase-1 encoder at initialisation (B = 0 for LoRA / LoRMA+, l = 1 for IA3).
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a = enc.eval()(_ids, attention_mask=_m); b = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        enc.train(); del ref
    elif METHOD == "norm_only":
        for p in enc.parameters(): p.requires_grad = False
        n = 0
        for m in enc.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.4f}M = {tr:,} ({100*tr/tot:.3f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model


def class_weights(train_docs):
    cnt = np.zeros(NL)
    for d in train_docs: cnt += np.bincount(d["y"], minlength=NL)
    w = np.ones(NL)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

In [12]:
# Cell 18 — Inference: windows -> probabilities -> weighted average per sentence
@torch.no_grad()
def predict_probs(model, docs, batch_size=None):
    """Per-doc [n_sentences, NL] probabilities. A sentence covered by several windows gets the weighted average of its probabilities
    (weight = 1 + distance (in sentences) to the nearest window edge when EDGE_WEIGHT, else 1)."""
    model.eval()
    bs = batch_size or (16 if CONTEXT_MODE == "window" else 128)
    jobs = [(di, a, b) for di, d in enumerate(docs) for (a, b) in doc_windows(d)]
    jobs.sort(key=lambda j: sum(len(t) for t in docs[j[0]]["tok"][j[1]:j[2]]))        # similar lengths together
    acc = [np.zeros((len(d["tok"]), NL), dtype=np.float32) for d in docs]
    wsum = [np.zeros(len(d["tok"]), dtype=np.float32) for d in docs]
    for i in range(0, len(jobs), bs):
        ids, mask, seg, _, owners, n_sent = build_batch(jobs[i:i + bs], docs, with_labels=False)
        ids, mask, seg = (torch.from_numpy(x).to(device) for x in (ids, mask, seg))
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(ids, mask, seg, n_sent)
        p = logits.float().softmax(-1).cpu().numpy()
        for g, (di, j, a, b) in enumerate(owners):
            w = 1.0 + min(j - a, b - 1 - j) if EDGE_WEIGHT else 1.0
            acc[di][j] += w * p[g]; wsum[di][j] += w
    return [acc[i] / np.maximum(wsum[i], 1e-8)[:, None] for i in range(len(docs))]

In [13]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def make_epoch_items(train_docs, rng):
    items = []
    for di, d in enumerate(train_docs):
        items += [(di, a, b) for a, b in doc_windows(d, rng)]
    rng.shuffle(items)
    return items

def train_rr(model, train_docs, val_docs, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, counts = class_weights(train_docs)
    print("class counts:", dict(zip(LABELS, counts.astype(int).tolist())))
    print("class weights:", dict(zip(LABELS, np.round(cw.cpu().numpy(), 2).tolist())))
    head_p = [p for n, p in model.named_parameters() if n.startswith("head.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("head.") or not p.requires_grad: continue
        (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=head_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(make_epoch_items(train_docs, random.Random(SEED + 1)))
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} sequences/epoch | {steps_per_epoch} optimiser steps/epoch | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=-1.0, epoch=0, state=None)
    t_train = time.time()
    for ep in range(1, epochs + 1):
        items = make_epoch_items(train_docs, random.Random(SEED + ep))
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(len(items) / BATCH)
        for bi in range(nb):
            chunk = items[bi * BATCH:(bi + 1) * BATCH]
            ids, mask, seg, y, _, n_sent = build_batch(chunk, train_docs)
            ids, mask, seg, y = (torch.from_numpy(x).to(device) for x in (ids, mask, seg, y))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(ids, mask, seg, n_sent)
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25*ACCUM:]):.4f} | lr x{m:.3f} | {time.time()-t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            ev = evaluate(val_docs, predict_probs(model, val_docs))
            rec.update(val_select=ev["select"], val_macro_f1_pooled=ev["ALL"]["macro_f1"], val_acc=ev["ALL"]["accuracy"])
            for dom in DOMAINS:
                if dom in ev: rec[f"val_{dom}_macro_f1"] = ev[dom]["macro_f1"]
            if ev["select"] > best["score"]:
                best = dict(score=ev["select"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(score=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [14]:
# Cell 20 — Build data, build the model, train
seed_everything(SEED)
if FINAL_FIT:
    train_docs = [d for dom in DOMAINS for d in (TRAIN_ALL[dom] + DEV_ALL[dom])]; val_docs = []
else:
    train_docs = [d for dom in DOMAINS for d in TRAIN_ALL[dom]]
    val_docs = [d for dom in DOMAINS for d in (DEV_ALL[dom][:SMOKE_VAL_DOCS] if SMOKE_TEST else DEV_ALL[dom])]
for dom in DOMAINS:
    tokenize_docs(TRAIN_ALL[dom], f"train_{dom}")
    tokenize_docs(DEV_ALL[dom], f"dev_{dom}")
token_stats(train_docs, "train")
if val_docs: token_stats(val_docs, "dev")
n_win = sum(len(doc_windows(d)) for d in train_docs)
print(f"train docs {len(train_docs)} | dev docs {len(val_docs)} | {n_win} sequences per epoch ({CONTEXT_MODE} mode)")

# reference point: always predicting the most frequent training label
if val_docs:
    _maj = int(np.bincount(np.concatenate([d["y"] for d in train_docs]), minlength=NL).argmax())
    _ev = evaluate(val_docs, [np.eye(NL, dtype=np.float32)[np.full(len(d["y"]), _maj)] for d in val_docs])
    print(f"majority-class baseline ('{LABELS[_maj]}'): dev macro-F1 per-domain mean {_ev['select']:.4f} | accuracy {_ev['ALL']['accuracy']:.3f}")

model = build_rr_model()
history, best, stats = train_rr(model, train_docs, val_docs, epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | dev select-score {best['score']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

tokenise train_CL:   0%|          | 0/40 [00:00<?, ?it/s]

tokenise dev_CL:   0%|          | 0/5 [00:00<?, ?it/s]

tokenise train_IT:   0%|          | 0/40 [00:00<?, ?it/s]

tokenise dev_IT:   0%|          | 0/5 [00:00<?, ?it/s]

train: 16845 sentences | tokens/sentence mean 42.9, median 34, p95 100, max 1199 | truncated at 160: 1.30%
dev: 2142 sentences | tokens/sentence mean 42.4, median 35, p95 96, max 417 | truncated at 160: 1.35%
train docs 80 | dev docs 10 | 1933 sequences per epoch (window mode)
majority-class baseline ('RatioOfTheDecision'): dev macro-F1 per-domain mean 0.0292 | accuracy 0.188
IA3 applied: 12 output-scaling vectors (keys/values) + 6 FFN-hidden vectors | sites=('k', 'v', 'ff')
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 35.68M | trainable 0.0213M = 21,261 (0.060%)
class counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
class weights: {'Fact': 1.0199999809265137, 'Issue': 6.340000152587891, 'ArgumentPetition

In [15]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    probs = predict_probs(model, val_docs); ev = evaluate(val_docs, probs)
    n_pred_labels = len(set(np.concatenate([p.argmax(1) for p in probs]).tolist()))
    print(f"loss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | dev select-score {ev['select']:.4f} | "
          f"predicted labels in use: {n_pred_labels}/{NL}")
    ok = drop < 0.9 and n_pred_labels >= 3
    print("SMOKE VERDICT:", "OK, the pipeline learns -> run the full job" if ok else
          "NOT OK: loss did not fall clearly or one label dominates; with ~250 steps an adapter can be slow, so compare with METHOD='full' before debugging")
else:
    print("not a smoke run")

not a smoke run


In [16]:
# Cell 22 — Dev analysis: per-domain scores, per-label table, confusions, bias tuning (full runs with a dev set)
BIAS = np.zeros(NL) if BIAS_FINAL is None else np.asarray(BIAS_FINAL, dtype=np.float64)
dev_ev = dev_ev_tuned = None; bias_gain = 0.0
if not SMOKE_TEST and val_docs:
    probs_val = predict_probs(model, val_docs)
    dev_ev = evaluate(val_docs, probs_val)
    for key in [*DOMAINS, "ALL"]:
        s = dev_ev[key]
        print(f"\n===== {key} | macro-F1 {s['macro_f1']:.4f} (gold-labels-only {s['macro_f1_gold']:.4f}) | macro-P {s['macro_p']:.4f} | "
              f"macro-R {s['macro_r']:.4f} | accuracy {s['accuracy']:.4f}")
        print(f"  {'label':24s} {'support':>8s} {'P':>7s} {'R':>7s} {'F1':>7s}")
        for i, l in enumerate(LABELS):
            print(f"  {l:24s} {int(s['support'][i]):8d} {s['P'][i]:7.3f} {s['R'][i]:7.3f} {s['F'][i]:7.3f}")
        print("  most frequent confusions (gold -> predicted):", [f"{g}->{p} x{n}" for n, g, p in top_confusions(s["cm"], 8)])
    print(f"\nselect-score (mean of per-domain macro-F1), UNTUNED: {dev_ev['select']:.4f}   <- report this one")
    if BIAS_TUNE:
        bias_t, b0, b1 = tune_bias(val_docs, probs_val)
        cv_gain = cv_bias_gain(val_docs, probs_val) if len(val_docs) >= 4 else 0.0
        bias_gain = cv_gain
        print(f"bias tuning on dev: {b0:.4f} -> {b1:.4f} (in-sample gain {b1 - b0:+.4f}, optimistic) | "
              f"2-fold document-level held-out gain {cv_gain:+.4f} | bias = {np.round(bias_t, 2).tolist()}")
        if cv_gain >= BIAS_MIN_GAIN:
            BIAS = bias_t; dev_ev_tuned = evaluate(val_docs, probs_val, BIAS)
            print(f"  kept: held-out gain >= BIAS_MIN_GAIN={BIAS_MIN_GAIN}. For FINAL_FIT=True set  BIAS_FINAL = {np.round(BIAS, 2).tolist()}")
        else:
            print(f"  held-out gain below BIAS_MIN_GAIN={BIAS_MIN_GAIN}: bias NOT used")
else:
    print("no dev analysis (smoke run or FINAL_FIT)" + ("" if BIAS_FINAL is None else f" | using BIAS_FINAL={np.round(BIAS, 2).tolist()}"))


===== CL | macro-F1 0.3231 (gold-labels-only 0.3878) | macro-P 0.3172 | macro-R 0.3546 | accuracy 0.5064
  label                     support       P       R      F1
  Fact                          249   0.502   0.574   0.536
  Issue                           3   0.000   0.000   0.000
  ArgumentPetitioner            344   0.596   0.494   0.541
  ArgumentRespondent            236   0.551   0.415   0.473
  Statute                        58   0.510   0.879   0.646
  Dissent                         0   0.000   0.000   0.000
  PrecedentReliedUpon           142   0.357   0.246   0.292
  PrecedentNotReliedUpon          0   0.000   0.000   0.000
  PrecedentOverruled              0   0.000   0.000   0.000
  RulingByLowerCourt             56   0.371   0.232   0.286
  RatioOfTheDecision            288   0.495   0.635   0.556
  RulingByPresentCourt           18   0.424   0.778   0.549
  None                            2   0.000   0.000   0.000
  most frequent confusions (gold -> predicted): ['Argu

In [17]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(docs, probs, bias, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    sub = {}
    for d, p in zip(docs, probs):
        pred = [LABELS[i] for i in (np.log(np.maximum(p, 1e-8)) + bias).argmax(1)]
        assert len(pred) == len(d["sents"]), f"{d['id']}: {len(pred)} predictions for {len(d['sents'])} sentences"
        assert d["id"] not in sub, f"document id {d['id']!r} occurs in more than one domain; one prediction.json cannot hold both"
        sub[d["id"]] = pred
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")          # must sit at the ROOT of the zip
    n = sum(len(v) for v in sub.values())
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} docs | {n} sentences | predicted labels:",
          dict(Counter(l for v in sub.values() for l in v).most_common()))
    verify_submission(out_dir / "submission.zip", docs)
    return sub

def verify_submission(zip_path, docs):
    """Re-open the zip like the evaluator would: root file name, ids, one official label per sentence, same order and length."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, dict), "prediction.json must be a JSON object {doc_id: [label, ...]}"
    want = {d["id"]: len(d["sents"]) for d in docs}
    missing, unknown = set(want) - set(sub), set(sub) - set(want)
    assert not missing and not unknown, f"ids differ from the Phase-3 test set: {len(missing)} missing, {len(unknown)} unknown"
    for did, lst in sub.items():
        assert isinstance(lst, list) and len(lst) == want[did], f"{did}: need {want[did]} labels, got {len(lst) if isinstance(lst, list) else lst!r}"
        assert all(l in L2I for l in lst), f"{did}: label outside label_vocab.json: {sorted(set(lst) - set(L2I))}"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} documents | first ids {list(sub)[:3]}")
    print("   -> upload exactly THIS file to the Phase-3 Codabench page (earlier phases produced files with the same name).")

if not SMOKE_TEST:
    for dom in DOMAINS: tokenize_docs(TEST_ALL[dom], f"test_{dom}")
    test_docs = [d for dom in DOMAINS for d in TEST_ALL[dom]]
    token_stats(test_docs, "test")
    probs_test = predict_probs(model, test_docs)
    submission = write_submission(test_docs, probs_test, BIAS, RUN_DIR / "submission")
    for dom in DOMAINS:
        c = Counter(l for d, p in zip(test_docs, probs_test) if d["domain"] == dom for l in (np.log(np.maximum(p, 1e-8)) + BIAS).argmax(1))
        print(f"  {dom} predicted label share:", {LABELS[k]: round(v / sum(c.values()), 3) for k, v in c.most_common()})
else:
    print("smoke run: no submission written")

tokenise test_CL:   0%|          | 0/5 [00:00<?, ?it/s]

tokenise test_IT:   0%|          | 0/5 [00:00<?, ?it/s]

test: 2197 sentences | tokens/sentence mean 43.5, median 34, p95 104, max 669 | truncated at 160: 1.64%
wrote /kaggle/working/runs/p3_encdec_ia3/submission/submission.zip | 10 docs | 2197 sentences | predicted labels: {'RatioOfTheDecision': 779, 'Fact': 347, 'Statute': 239, 'PrecedentReliedUpon': 235, 'ArgumentPetitioner': 223, 'RulingByLowerCourt': 124, 'PrecedentNotReliedUpon': 78, 'RulingByPresentCourt': 56, 'ArgumentRespondent': 46, 'None': 33, 'Issue': 25, 'Dissent': 12}
VERIFIED /kaggle/working/runs/p3_encdec_ia3/submission/submission.zip | 10 documents | first ids ['HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', 'COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224', 'SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC']
   -> upload exactly THIS file to the Phase-3 Codabench page (earlier phases produced files with the same name).
  CL predicted label share: {'RatioOfTheDecision': 

In [18]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full / context tables)
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o

def _ev_summary(ev):
    if ev is None: return None
    return {k: (dict(macro_f1=v["macro_f1"], macro_f1_gold=v["macro_f1_gold"], macro_p=v["macro_p"], macro_r=v["macro_r"], accuracy=v["accuracy"],
                     per_label={LABELS[i]: dict(support=int(v["support"][i]), P=round(float(v["P"][i]), 4), R=round(float(v["R"][i]), 4),
                                                F1=round(float(v["F"][i]), 4)) for i in range(NL)},
                     confusion=v["cm"].tolist(), top_confusions=top_confusions(v["cm"], 12)) if isinstance(v, dict) else v)
            for k, v in ev.items()}

log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, context_mode=CONTEXT_MODE, window_tokens=WINDOW_TOKENS, window_overlap=WINDOW_OVERLAP,
           max_sent_tok=MAX_SENT_TOK, edge_weight=EDGE_WEIGHT, domains=DOMAINS, domain_tag=DOMAIN_TAG, final_fit=FINAL_FIT, smoke=SMOKE_TEST,
           encoder_causal=lm_cfg.encoder_causal, enc_layers=lm_cfg.n_encoder_layers, ia3_sites=list(IA3_SITES), pos_type=lm_cfg.pos_type, hp=HP[METHOD], epochs=EPOCHS, batch=BATCH, accum=ACCUM,
           class_weight=CLASS_WEIGHT, labels=LABELS, params_total=model.n_total, params_trainable=model.n_trainable,
           history=history, best_epoch=best["epoch"], stats=stats, bias=BIAS.tolist(), bias_gain=bias_gain,
           dev_untuned=_ev_summary(dev_ev), dev_tuned=_ev_summary(dev_ev_tuned))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "labels": LABELS, "log": _jsonable(log)},
           RUN_DIR / f"rr_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['rr_p3_encdec_ia3.pt', 'run_log.json', 'submission']


In [19]:
# Cell 25 — Export this run's per-sentence distributions ("soft labels") so it can serve as the KD teacher (run #4)
# All four Phase-1 models share one tokenizer and the label list is fixed, so these arrays can teach ANY student without this
# notebook's model code. Stored as float16 log-probabilities: softmax(logp / T) = softmax(z / T) because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time()
    _exp = {"train": [d for d in train_docs], "dev": [d for d in val_docs]}
    _out, _split = {}, {}
    for _name, _docs in _exp.items():
        if not _docs: continue
        for d, p in zip(_docs, predict_probs(model, _docs)):
            _out[d["uid"]] = np.log(np.maximum(p, 1e-8)).astype(np.float16); _split[d["uid"]] = _name
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, context_mode=CONTEXT_MODE, window_tokens=WINDOW_TOKENS,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable), epochs=int(EPOCHS), best_epoch=int(best["epoch"]),
                 final_fit=FINAL_FIT, dev_select=None if dev_ev is None else float(dev_ev["select"]))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, labels=list(LABELS), docs=_out, split=_split), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.0f} MB) in {time.time()-_t0:.0f}s | {len(_out)} documents | teacher params "
          f"{_meta['params_total']/1e6:.1f}M | dev select-score {_meta['dev_select']}")
    print("caveat: on the TRAIN documents the teacher is overconfident (it was trained on them); its 'dev' distributions are out-of-sample.")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)


## What to do next

1. **Smoke test first:** `SMOKE_TEST=True` (a few minutes). Read the *masked-LM loss* line (proves the encoder loaded), the *identity-at-init* line (IA³ must equal Phase-1), the trainable-parameter count (about 21k for IA³ + head), the majority-class baseline and the smoke verdict. With only about 21k trainable parameters, 250 steps is short: judge IA³ by the **trend of the loss**, and compare with `METHOD="full"` on the smoke run if in doubt.
2. **Full run:** `SMOKE_TEST=False`, defaults. It selects the epoch on dev (mean of the CL and IT macro-F1) and **already writes a valid `submission.zip`**: upload it to Codabench right away; everything below is refinement.
3. **Controls for the report** (change Cell 2 only, new `RUN_NAME` each time):
   * technique on the same encoder: `METHOD="lora"` / `"lorma"` (how much do the extra trainable parameters buy?) and `METHOD="full"` (reference: *can PEFT approach full adaptation?*);
   * IA³ sites: `IA3_SITES=("k","v")` vs `("ff",)` vs all three (which activations matter?);
   * IA³ learning rate: if it underfits, try `lr=1e-2`; if the loss is unstable, `1e-3`;
   * context: `CONTEXT_MODE="sentence"` and a smaller `WINDOW_TOKENS` (256) against 700.
4. **Final fit (optional):** `FINAL_FIT=True`, `EPOCHS=<best epoch>`, `BIAS_FINAL=<list printed by Cell 22>`.
5. **KD:** not as teacher (35.66M < 36.58M student). It can still serve a smaller student; otherwise export from a decoder run.
6. Copy every `run_log.json` into the report folder (parameters, peak GPU memory, training time, per-label P/R/F1, confusions for CL and IT).